In [ ]:
from parserrtm.longwave import InputLW
from parserrtm.shortwave import InputSW
from parserrtm.runner import Runner, read_output

from pathlib import Path
from sys import platform

## 1. Validate parser on longwave examples

### 1.1 Read RRTM_LW input files

In [ ]:
#test case input files distributed with RRTM_LW
# structure is "case name: [input_rrtm, in_cld_rrtm]"
# in_cld_rrtm is not used if example is a clear-sky calculation
examples = {
    'ICRCCM_sonde':['input_rrtm_ICRCCM_sonde'],
    'MLS':['input_rrtm_MLS'],
    'MLW':['input_rrtm_MLW'],
    'SAW':['input_rrtm_SAW'],
    'TROP':['input_rrtm_TROP'],
    'MLS-xsec':['input_rrtm_MLS-xsec'],
    'MLS-cld1':['input_rrtm_MLS-cld', 'in_cld_rrtm_MLS-cld1'],
    'MLS-cld2':['input_rrtm_MLS-cld', 'in_cld_rrtm_MLS-cld2'],
    'MLS-cld3':['input_rrtm_MLS-cld', 'in_cld_rrtm_MLS-cld3'],
    'sgp_20000313.172900':['input_rrtm_sgp_20000313.172900','in_cld_rrtm-sgp_20000313.172900'],
    'sgp_20000313.203000':['input_rrtm_sgp_20000313.203000','in_cld_rrtm-sgp_20000313.203000'],
    'MLS1-streamer_param':['input_rrtm_MLS1-cld_disort','in_cld_rrtm_MLS1-streamer_param'],
    'MLS1-fu_param':['input_rrtm_MLS1-cld_disort','in_cld_rrtm_MLS1-fu_param']
}

#read test case files into parserrtm.InputLW objects
parsed_examples = { }
folder = Path('../rrtm_lw/run_examples')
for case_name, input_files in examples.items():
    #prepend folder location before each file
    input_files = [folder/name for name in input_files]
    #parse case input files
    input_lw = InputLW(input_files)
    parsed_examples[case_name] = input_lw
    #print out some case information
    print(case_name)
    print(f'CXID: {input_lw.CXID}')
    print(f"IATM: {input_lw.IATM}, IXSECT: {input_lw.IXSECT}, ISCAT: {input_lw.ISCAT}")
    print(f"NUMANGS: {input_lw.NUMANGS}, IOUT: {input_lw.IOUT}, ICLD: {input_lw.ICLD}")
    print(f"TBOUND: {input_lw.TBOUND}, IEMIS: {input_lw.IEMIS}, IREFLECT: {input_lw.IREFLECT}")
    print(f"SEMISS(1): {input_lw['SEMISS(1)']} ...")
    print('---------')

### 1.2 Run RRTM on parsed input files

In [ ]:
if platform == 'darwin':
    #macOS using lima as a linux virtual machine,
    # where machine is mounted at ~/linuxvm
    kwargs = {
            # where input and output files are stored
            'tmp_path': '/private/tmp/lima',
            # shell in which to run the RRTM processes
            'shell': '/usr/local/bin/lima sh -c',
            # path to RRTM executable
            'exec_path': Path('../rrtm_lw/rrtm_v3.3_linux_pgf90').resolve(),
            # increase workers for parallel processing
            'n_workers': 1
        }
else:
    #linux options
    kwargs = {
            'tmp_path': '/tmp',
            'shell': '/bin/sh -c',
            'exec_path':Path('../rrtm_lw/rrtm_v3.3.1_linux_pgf90').resolve(),
            'n_workers': 1
        }
if platform != 'linux':
    print('Please run a linux VM or compile RRTM for your system! \
Then change exec_path to the path of your newly-compiled executable')

#start client
client = Runner(**kwargs)

#run cases
case_order = sorted(examples.keys())
inputs     = [parsed_examples[case] for case in case_order]
outputs, logs = client.run(inputs,verbose=False)

### 1.3 Compare outputs

In [ ]:
for new_output,case in zip(outputs,case_order):
    orig_output = read_output(folder/f'output_rrtm_{case}')
    error = new_output['net_flux']-orig_output['net_flux']
    print(f"net_flux avg error: {error.mean('level'):1.1e} W/m^2")

## 2. Validate parser on shortwave examples

### 2.1 Read RRTM_SW input files

In [ ]:
#test case input files distributed with RRTM_SW
# structure is "case name: [input_rrtm, in_cld_rrtm, in_aer_rrtm]"
examples = {
    'trp':['input_rrtm_sw_trp'],
    'trp_aerosol':['input_rrtm_sw_trp_aerosol', 'none', 'in_aer_rrtm_trp_aerosol'],
    'trp_icecld':['input_rrtm_sw_trp_cld','in_cld_rrtm_trp_icecld'],
    'trp_mixcld':['input_rrtm_sw_trp_cld','in_cld_rrtm_trp_mixcld'],
    'trp_layervals':['input_rrtm_sw_trp_layervals'],
    'mls':['input_rrtm_sw_mls'],
    'saw':['input_rrtm_sw_saw'],
    'pressure_levels':['input_rrtm_sw_pressure_levels'],
}

#read test case files into parserrtm.InputSW objects
parsed_examples = { }
folder = Path('../rrtm_sw/example_runs')
for case_name, input_files in examples.items():
    #prepend folder location before each file
    input_files = [folder/name if name != 'none' else name for name in input_files]
    #parse case input files
    input_lw = InputSW(input_files)
    parsed_examples[case_name] = input_lw
    #print out some case information
    print(case_name)
    print(f'CXID: {input_lw.CXID}')
    print(f"IATM: {input_lw.IATM}, ICLD: {input_lw.ICLD}, IAER: {input_lw.IAER}")
    print('---------')

### 2.2 Run RRTM_SW on parsed input files

In [ ]:
if platform == 'darwin':
    #macOS using lima as a linux virtual machine,
    # where machine is mounted at ~/linuxvm
    kwargs = {
            # where input and output files are stored
            'tmp_path': '/private/tmp/lima',
            # shell in which to run the RRTM processes
            'shell': '/usr/local/bin/lima sh -c',
            # path to RRTM executable
            'exec_path': Path('../rrtm_sw/rrtm_sw_v2.7.3_linux_pgf90').resolve(),
            # increase workers for parallel processing
            'n_workers': 1
        }
else:
    #linux options
    kwargs = {
            'tmp_path': '/tmp',
            'shell': '/bin/sh -c',
            'exec_path':Path('../rrtm_sw/rrtm_sw_v2.7.3_linux_pgf90').resolve(),
            'n_workers': 1
        }
if platform != 'linux':
    print('Please run a linux VM or compile RRTM for your system! \
Then change exec_path to the path of your newly-compiled executable')

#start client
client = Runner(**kwargs)

#run cases
case_order = sorted(examples.keys())
inputs     = [parsed_examples[case] for case in case_order]
outputs, logs = client.run(inputs,verbose=True)

### 1.3 Compare outputs

In [ ]:
print("case name \t net err [%] ")
for new_output,case in zip(outputs,case_order):
    orig_output = read_output(folder/f'output_rrtm_sw_{case}')
    error = new_output['net_flux']-orig_output['net_flux']
    print(f"{case} \t {error.mean('level'):.2e}")

# -- END OF NOTEBOOK ---

In [ ]:
import numpy as np
widths=[6,10,14,14,15,14,15,13]
slices = np.cumsum([0,*widths])
for i in range(len(slices)-1):
    print(lines[start_i+5][slices[i]:slices[i+1]])

In [ ]:
lines[start_i]

In [ ]:
!cat '/private/tmp/lima/rrtm8669_worker00/OUTPUT_RRTM'

In [ ]:
ds

In [ ]:
outputs[0]

In [ ]:
case_order

In [ ]:
inputs[4].get_explicit_record_order('in_aer_rrtm')

In [ ]:
inputs[4].